# 🚀 TikTok 4.5B Viral Intelligence & Analytics Pipeline
### 289GB 허깅페이스 원본(`kuben-developer/tiktok-videos-4b`) 초고속 스트리밍 분석

> **💡 289GB를 내 컴퓨터에 다운로드받지 마세요!**  
> Parquet 포맷과 **DuckDB HTTPFS**를 활용하면, 수백 기가의 파일을 다운로드하지 않고도 허깅페이스 클라우드에 올려진 45억 건의 데이터에서 필요한 컬럼과 집계 데이터만 **원격 스트리밍 쿼리**할 수 있습니다.

## 1. 분석 환경 및 DuckDB 엔진 세팅

In [ ]:
!pip install -q duckdb polars plotly huggingface_hub

import duckdb
import polars as pl
import plotly.express as px
import json

# DuckDB 인메모리 엔진 가동 및 원격 HTTPFS 확장 모듈 로드
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
print("✅ DuckDB HTTPFS 원격 쿼리 엔진 초기화 완료!")

## 2. 허깅페이스 289GB 데이터셋 구조 탐색 (무다운로드 메타데이터 스캔)
27개의 분할 Parquet 파일 중 0번 파일(`videos-00.parquet`, 약 1.67억 건)의 스키마를 즉시 원격 스트리밍으로 읽어옵니다.

In [ ]:
# 허깅페이스 공식 원본 Parquet URL (videos-00.parquet)
parquet_url = "https://huggingface.co/datasets/kuben-developer/tiktok-videos-4b/resolve/main/videos-00.parquet"

print("[*] 허깅페이스 원격 Parquet 스키마 조회 중...")
schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_url}')").fetchdf()
display(schema_df)

## 3. 실전 쿼리 1: 틱톡 영상 샘플 및 바이럴 인게이지먼트 분석
조회수, 좋아요, 공유수, 북마크(saves), 캡션 및 사용 음원을 즉시 추출합니다.

In [ ]:
query_sample = f"""
SELECT 
    content_id,
    views,
    likes,
    shares,
    saves,
    comments,
    round((likes * 100.0) / NULLIF(views, 0), 2) AS like_rate_pct,
    music_title,
    substr("desc", 1, 80) AS caption_preview,
    country,
    language,
    create_time
FROM read_parquet('{parquet_url}')
WHERE views IS NOT NULL
LIMIT 25;
"""

print("[*] 허깅페이스 원격 데이터 스트리밍 쿼리 실행 중...")
sample_df = con.execute(query_sample).fetchdf()
display(sample_df)

## 4. 실전 쿼리 2: 바이럴을 주도한 최다 사용 BGM 음원(Music ID) 랭킹
틱톡 알고리즘 바이럴의 핵심인 배경음악(BGM)별 사용 빈도와 누적 조회수 점유율을 분석합니다.

In [ ]:
query_top_music = f"""
SELECT 
    music_id,
    music_title,
    count(*) AS video_usage_count,
    sum(views) AS total_music_views,
    avg(likes) AS avg_likes_per_video
FROM read_parquet('{parquet_url}')
WHERE music_title IS NOT NULL AND music_title != ''
GROUP BY 1, 2
ORDER BY video_usage_count DESC
LIMIT 15;
"""

top_music_df = con.execute(query_top_music).fetchdf()
display(top_music_df)

fig_music = px.bar(
    top_music_df,
    x='video_usage_count',
    y='music_title',
    orientation='h',
    color='total_music_views',
    title='🎵 틱톡 최다 바이럴 배경음악(BGM) 탑 15 및 누적 조회수'
)
fig_music.show()

## 5. [선택 사항] 구글 드라이브로 영구 소장 백업 (300GB 이상 계정용)
만약 구글 드라이브 잔여 용량이 300GB 이상이라면, 아래 셀로 허깅페이스 원본을 내 드라이브에 직접 저장할 수 있습니다.

In [ ]:
# from google.colab import drive
# from huggingface_hub import snapshot_download

# drive.mount('/content/drive')
# snapshot_download(
#     repo_id='kuben-developer/tiktok-videos-4b',
#     repo_type='dataset',
#     local_dir='/content/drive/MyDrive/TikTok_4B_Dataset'
# )
# print('✅ 구글 드라이브 백업 완료!')